# 09 — Cross-module consistency

**Code under test:** `baseline pipeline + webapp/app.py`

**Purpose:** Check that energy, LCOE, and capacity metrics agree across layers.

Run cells **top to bottom**. Each markdown cell explains the **next code cell** — what it does, what to inspect in the output, and what counts as a pass.

Track overall audit progress in Obsidian (`FlexiMORP Calculation Audit.md`). These notebooks are the lab workbook, not the checklist.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("error", category=RuntimeWarning)


def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "fleximorpv2").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find fleximorp-project root. "
        "Open Jupyter from the repo or a notebooks/ subdirectory."
    )


_repo = _repo_root()
_audit = _repo / "notebooks" / "audit"
for path in (_repo, _audit):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from _audit_helpers import (
    REPO_ROOT,
    OUTPUT_DIR,
    SITE_OUTPUT_DIR,
    reload_fleximorp,
    assert_close,
    assert_energy_balance,
    assert_cf_bounds,
    assert_positive,
)

reload_fleximorp()
np.random.seed(42)
print(f"Repo root: {REPO_ROOT}")
print(f"Audit outputs: {OUTPUT_DIR}")
print(f"Site outputs: {SITE_OUTPUT_DIR}")

## Step 1 — Energy balance on baseline result

**Run the next cell.**

**Pass if:** `annual_energy ≈ capacity_factor × total_capacity × 8760` (within 5%).

If this fails with `annual_energy=0`, the bug is in technology/economic coupling — log it in Obsidian findings.

In [ ]:
from fleximorpv2.config import load_config
from fleximorpv2.baseline_optimization import BaselineOptimization

config = load_config("alaska")
config.uncertainty["monte_carlo_runs"] = 10
results = BaselineOptimization(config).optimize("production", 200_000, method="scipy")

assert_energy_balance(
    results.technical_metrics["annual_energy"],
    results.technical_metrics["total_capacity"],
    results.technical_metrics["capacity_factor"],
)
assert_cf_bounds(results.technical_metrics["capacity_factor"])
print("Financial:", results.financial_metrics)
print("Technical:", results.technical_metrics)

## Step 2 — Manual LCOE cross-check

**Run the next cell** after computing LCOE from CAPEX, OPEX, and energy independently.

**Pass if:** manual LCOE within ~10% of `results.financial_metrics['lcoe']` (wider tolerance until cost stack is validated in notebook 06).

In [ ]:
# TODO: pull capex/opex/energy from results or cost_breakdown and compare LCOE
manual_lcoe = None  # £/MWh from notebook 06 logic
reported = results.financial_metrics["lcoe"]
print(f"Reported LCOE: {reported:.2f} £/MWh")
if manual_lcoe is not None:
    assert_close(reported, manual_lcoe, label="LCOE cross-check", rtol=0.10)